# Õppetund 12 - Vestluse ajaloo vähendamine agendi märkmikuga

See märkmik demonstreerib, kuidas hallata konteksti pikkades vestlustes, kasutades Microsoft Agent Frameworki. Vestluste kasvades suureneb tokenite arv — lõpuks ületades mudeli kontekstiakna suuruse. Seda lahendame **konteksti kokkuvõtte mustriga** ja **agendi märkmikuga** püsiva mälu jaoks.

## Mida sa õpid:
1. **Miks kontekstihaldus on oluline**: Tokenipiirangute ja kontekstiakende mõistmine
2. **Kontekstiteadlikud agendid**: Agentide loomine, kes haldavad oma vestluse konteksti
3. **Konteksti kokkuvõtte muster**: Tööriistade kasutamine vestluse ajaloo kokkusurumiseks
4. **Agendi märkmik**: Püsiv mälu, mis säilib ka konteksti vähendamisel

## Nõuded:
- Azure OpenAI seadistus koos keskkonnamuutujate konfigureerimisega
- Eelnevate õppetundide põhiteadmised agentidest


## Seadistamine


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Miks konteksti haldamine on oluline

Igal LLM-il on piiratud **kontekstiväli** — maksimaalne tokenite arv, mida ta suudab ühe päringu jooksul töödelda. Kui mitme vooruga vestlus edeneb:

- **Tokenite arv kasvab lineaarse kiirusega** iga kasutaja sõnumi ja assistendi vastuse järel.
- **Pääsupäästjate tokenid domineerivad kulu** tõttu, et kogu ajalugu saadetakse iga vooru järel uuesti.
- Lõpuks vestlus **ületab kontekstivälja** ja mudel kas kärbib selle või tekitab vea.

### Konteksti haldamise strateegiad

| Strateegia | Kuidas töötab | Kompromiss |
|---|---|---|
| **Kärpimine** | Vanimad sõnumid eemaldatakse | Kaob varasem kontekst |
| **Kokkuvõte** | Vanemad sõnumid kokku võetakse kokkuvõtteks | Mõned detailid kaovad, kuid olulised punktid jäävad alles |
| **Märkmik / Välimälu** | Olulised faktid salvestatakse vestlusest väljaspool | Nõuab tööriistade kutsumist, kuid säilib ka vähendamisel |

Selles märkmikus ühendame **kokkuvõtte** ja **märkmiku tööriista**, et agent saaks jätkata sujuvalt isegi siis, kui vestluse ajalugu kokku võetakse.


## Kontekstitundliku agendi loomine


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Pika vestluse simuleerimine

Vaatame läbi mitme sammulise vestluse, et näha, kuidas kontekst koguneb. Agent peaks hoidma olulisi üksikasju (eelistused, eelarve, reisi kuupäevad) kõigi sammuside jooksul ja näitama järjepidevust.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Pange tähele, kuidas agent säilitab konteksti varasematest vestlustest — ta teab Jaapanist, sushi'st, templidest, fotograafiast, $3000 eelarvest, üksikturismist ja aprilli ajastusest. Lühikeses vestluses töötab see hästi, kuid vestluse kasvades muutub kogu ajaloo uuesti saatmine kulukaks.

Jätkame vestlust veel mitme kirjaga, et näha, kuidas kontekst koguneb:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Konteksti Kokkuvõtmise Muster

Kui vestlus kasvab, saame kasutada **kokkuvõtmise tööriista**, et kogutud konteksti kokku pakkida kompaktseks vormiks. Agent kutsub seda tööriista, et salvestada olulised eelistused, nii et isegi kui vanemad sõnumid eemaldatakse, jääb põhiinfo alles.

See muster on arenenuma ajaloo vähendamise aluseks:
1. Agent tuvastab vestlusest olulised faktid
2. See kutsub kokkuvõtmise tööriista, et neid salvestada
3. Vanemad sõnumid võivad ohutult eemaldada, sest kokkuvõte kajastab olulist

Allpool määratleme tööriista `summarize_preferences`, mida agent saab kutsuda, et salvestada kompaktne kokkuvõte sellest, mida ta on õppinud.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Kokkuvõte

Selles õppetükis õppisid, kuidas hallata konteksti pikaajalistes agentide vestlustes, kasutades Microsoft Agent Frameworki:

### Põhimõisted
- **Kontekstaknad on piiratud** — iga vestluse ajaloo tokeni eest tuleb maksta ja need loevad limiiti.
- **Kokkuvõtte tööriistad** võimaldavad agendil kogunenud konteksti kokku võtta kompaktseteks kokkuvõteteks, vähendades tokenite kasutust ja säilitades olulise info.
- **Agendi märkmikud** pakuvad püsivat välist mälu, mis säilib vaatamata vestluse vähendamisele.

### Mida sa ehitasid
- **Kontekstitundliku agendi**, mis säilitab järjepidevuse mitme vooruline vestluste ajal
- **Kokkuvõtte tööriista** (`summarize_preferences`), mis salvestab olulisi kasutajaandmeid kompaktse vorminguna
- **Mitme vooruline vestlus**, mis demonstreerib konteksti säilitamist ja muutuste käsitlemist

### Reaalsed kasutusjuhtumid
- **Klienditeeninduse robotid**: Mäletavad eelistusi pikkade tugiseansside jooksul
- **Isiklikud assistendid**: Jälgivad käimasolevaid projekte konteksti ümber selgitamata
- **Õppejõud**: Säilitavad õpilase edenemist paljude suhtluste jooksul

### Järgmised sammud
- Rakendada täistoiminguga märkmikufunktsioon failipõhise püsivusega
- Lisada automaatne ajaloo kärpimine pärast kokkuvõtet
- Ühendada vektorandmebaasidega semantilise mälu otsinguks
- Ehitada agente, kes suudavad jätkata vestlusi päevade möödudes täiskontekstiga


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastutusest loobumine**:
See dokument on tõlgitud tehisintellektipõhise tõlketeenusega [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi püüame tagada täpsust, palun pidage meeles, et automaatsed tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada ametlikuks allikaks. Tähtsa teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta võimalike arusaamatuste või valesti mõistmiste eest, mis võivad tekkida selle tõlke kasutamisest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
